# **Building a RAG System with LangChain and FAISS**

Introduction to RAG. RAG comines the power of retrieval systems with generative AI models. Instead of relying solely on the model's training data. RAG:

1. Retrieves relevant documents from a knowledge base.
2. Uses these documents as context for the LLM.
3. Generates responses based on both the retrieved context and the model's knowledge.

## FAISS

FAISS is a library for efficient similarity search and clustering of dense vectors.

Key advantage:
1. Extremely fast similarity search.
2. Memory efficient.
3. Supports GPU acceleration.
4. Can handle millions of vectors.

How it works:

- Indexed vectors for fast nearest neighbor search.
- Returns most similar vectors based on distance metrics.

In [3]:
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
from typing import List, Dict, Any, Optional

# Langchain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# Langchain specific imports
from langchain.document_loaders import TextLoader, PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# load environment variables
load_dotenv()

True

In [ ]:
# Data Ingestion and Processing
temp_dir = "/var/folders/5l/5z3454654d5b9c9jrn1pkgjc0000gn/T/tmpofc4sb24"
loader = DirectoryLoader(
    temp_dir,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")
print(f"First document preview: ")
print(documents[0].page_content[:200] + "...")

Loaded 4 documents
First document preview: 

    Data Engineering and ETL Pipelines

    Data Engineering and ETL Pipelines form the backbone of data-driven organizations by enabling ingestion, transformation, storage, and delivery of reliable ...


In [10]:
# Text splitting
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len,
    separators=[" "],
)

document_chunks = text_splitter.split_documents(documents)

print(f"Created {len(document_chunks)} chunks from {len(documents)} documents")
print(f"\nChunk example: ")
print(f"\nContent: {document_chunks[0].page_content[:150]}...")
print(f"\nMetadata: {document_chunks[0].metadata}")

Created 20 chunks from 4 documents

Chunk example: 

Content: Data Engineering and ETL Pipelines

    Data Engineering and ETL Pipelines form the backbone of data-driven organizations by enabling ingestion, trans...

Metadata: {'source': '/var/folders/5l/5z3454654d5b9c9jrn1pkgjc0000gn/T/tmpofc4sb24/doc_3.txt'}


In [13]:
# load embedding models
embeddings = OpenAIEmbeddings(
    base_url=os.getenv("IQ_BASE_URL"),
    model=os.getenv("EMBEDDING_MODEL"),
    dimensions=1536,
)